In [0]:
# ==========================================
# 00_Config_and_Logging
# Shared by every pipeline notebook via:  %run ./00_Config_and_Logging
# One place for paths, the staging layout and the audit-log helper.
# ==========================================
import os
from datetime import datetime
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, TimestampType)

VOLUME_ROOT  = "/Volumes/workspace/default"
STAGING_ROOT = f"{VOLUME_ROOT}/staging"

# Staging layout
#   staging/new        files fetched but NOT yet loaded into Bronze
#   staging/processed  files already loaded into Bronze (moved here after success)
NEW_DIR       = f"{STAGING_ROOT}/new"
PROCESSED_DIR = f"{STAGING_ROOT}/processed"

BRONZE_ROOT = f"{VOLUME_ROOT}/bronze_layer"
SILVER_ROOT = f"{VOLUME_ROOT}/silver_layer"

bronze_delta_path = f"{BRONZE_ROOT}/bronze_carbon_data"
quarantine_path   = f"{BRONZE_ROOT}/bronze_quarantine"
silver_delta_path = f"{SILVER_ROOT}/silver_air_quality_readings"
log_table_path    = f"{STAGING_ROOT}/pipeline_execution_logs"   # unchanged

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze_layer")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver_layer")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.staging")
dbutils.fs.mkdirs(NEW_DIR)
dbutils.fs.mkdirs(PROCESSED_DIR)

print("new/       :", NEW_DIR)
print("processed/ :", PROCESSED_DIR)
print("bronze     :", bronze_delta_path)
print("quarantine :", quarantine_path)
print("log table  :", log_table_path)

In [0]:
# ==========================================
# Audit log table: one row per file / batch processed, in every layer
# ==========================================
log_schema = StructType([
    StructField("layer_processed",     StringType(),    True),  # Raw-to-Bronze / Bronze-to-Silver
    StructField("parameter_processed", StringType(),    True),  # file path or date window
    StructField("load_type",           StringType(),    True),  # full / incremental / backfill
    StructField("start_time",          TimestampType(), True),
    StructField("end_time",            TimestampType(), True),
    StructField("status",              StringType(),    True),  # Success / Partial Success / Failure
    StructField("rows_inserted",       IntegerType(),   True),
    StructField("rows_updated",        IntegerType(),   True),
    StructField("rows_quarantined",    IntegerType(),   True),
    StructField("error_message",       StringType(),    True),
])

try:
    existing_cols = spark.read.format("delta").load(log_table_path).columns
    if existing_cols != log_schema.fieldNames():
        raise Exception(
            "The log table still has the OLD column layout. Run this once, then re-run this cell:\n"
            f'    dbutils.fs.rm("{log_table_path}", recurse=True)')
    print("Log table already exists with the current layout.")
except Exception as e:
    if "OLD column layout" in str(e):
        raise
    spark.createDataFrame([], log_schema).write.format("delta").save(log_table_path)
    print("Audit log table created.")


def write_log(layer, parameter, load_type, start_time, status,
              rows_inserted=0, rows_updated=0, rows_quarantined=0, error_message=None):
    """Append one audit row. Call it from a `finally:` block so failures are logged too."""
    row = [(layer, parameter, load_type, start_time, datetime.now(), status,
            int(rows_inserted), int(rows_updated), int(rows_quarantined), error_message)]
    (spark.createDataFrame(row, schema=log_schema)
          .write.format("delta").mode("append").save(log_table_path))